# Medical Text Classification - End-to-End Project

## Project Overview
This notebook guides through the complete workflow for building a lightweight medical condition classifier using NLP, including:
- Dataset analysis and preprocessing
- Model training and evaluation
- API development with FastAPI
- Docker containerization
- CI/CD automation with GitHub Actions
- Monitoring with Prometheus and Grafana
- Latency optimization with ONNX

## 1. Environment Setup and Dependency Installation

In [ ]:
# Import required libraries
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("Libraries imported successfully!")
print(f"Python version: {sys.version}")

## 2. Data Loading and Dataset Profiling

In [ ]:
# Define data paths
data_dir = Path('../data/raw')
train_path = data_dir / 'medical_tc_train.csv'
test_path = data_dir / 'medical_tc_test.csv'
labels_path = data_dir / 'medical_tc_labels.csv'

# Load data
print("Loading training data...")
train_df = pd.read_csv(train_path)

print("Loading test data...")
test_df = pd.read_csv(test_path)

print("Loading labels...")
labels_df = pd.read_csv(labels_path)

print(f"\nDataset shapes:")
print(f"  Training set: {train_df.shape}")
print(f"  Test set: {test_df.shape}")
print(f"  Labels: {labels_df.shape}")

In [ ]:
# Inspect data structure
print("Training data columns:")
print(train_df.columns.tolist())
print(f"\nData types:\n{train_df.dtypes}")
print(f"\nMissing values:")
print(train_df.isnull().sum())
print(f"\nFirst 2 rows:")
train_df.head(2)

In [ ]:
# Inspect labels
print("Available disease conditions:")
print(labels_df)

# Create label mapping
label_map = dict(zip(labels_df['condition_label'], labels_df['condition_name']))
print(f"\nLabel Mapping: {label_map}")

## 3. EDA - Exploratory Data Analysis for Disease Descriptions

In [ ]:
# Class distribution analysis
print("Class Distribution:")
class_dist = train_df['condition_label'].value_counts().sort_index()
for label_id, count in class_dist.items():
    label_name = label_map[label_id]
    percentage = (count / len(train_df)) * 100
    print(f"  {label_name}: {count} ({percentage:.1f}%)")

print(f"\nTotal training samples: {len(train_df)}")
print(f"Number of classes: {len(label_map)}")

In [ ]:
# Text length analysis
train_df['text_length'] = train_df['medical_abstract'].str.len()
train_df['word_count'] = train_df['medical_abstract'].str.split().str.len()

print("Text Statistics:")
print(f"\nCharacter count:")
print(train_df['text_length'].describe())
print(f"\nWord count:")
print(train_df['word_count'].describe())

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution bar plot
class_names = [label_map[i] for i in sorted(class_dist.index)]
axes[0].bar(class_names, class_dist.values, color='steelblue')
axes[0].set_title('Class Distribution in Training Set', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Samples')
axes[0].tick_params(axis='x', rotation=45)

# Text length distribution
axes[1].hist(train_df['word_count'], bins=30, color='darkgreen', alpha=0.7, edgecolor='black')
axes[1].set_title('Distribution of Text Length (Words)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of Words')
axes[1].set_ylabel('Frequency')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/eda_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("Visualization saved!")

In [ ]:
# Text length by class
fig, ax = plt.subplots(figsize=(12, 5))

for label_id in sorted(label_map.keys()):
    class_texts = train_df[train_df['condition_label'] == label_id]['word_count']
    ax.hist(class_texts, bins=20, alpha=0.6, label=label_map[label_id])

ax.set_title('Text Length Distribution by Disease Class', fontsize=12, fontweight='bold')
ax.set_xlabel('Number of Words')
ax.set_ylabel('Frequency')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/eda_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Text Preprocessing and Feature Engineering

In [ ]:
# Import preprocessing utilities
sys.path.insert(0, '..')
from src.utils.preprocessing import TextPreprocessor, preprocess_batch

# Initialize preprocessor
preprocessor = TextPreprocessor()

# Sample text before and after preprocessing
sample_text = train_df['medical_abstract'].iloc[0]
print("Original text:")
print(sample_text[:200] + "...")
print(f"\nLength: {len(sample_text)} characters")

processed_text = preprocessor.preprocess(sample_text)
print("\n" + "="*50)
print("Processed text:")
print(processed_text[:200] + "...")
print(f"\nLength: {len(processed_text)} characters")

In [ ]:
# Preprocess all training texts
print("Preprocessing all training texts...")
train_texts_processed = preprocess_batch(train_df['medical_abstract'].values, preprocessor)
train_df['processed_text'] = train_texts_processed

print("Preprocessing complete!")
print(f"Processed {len(train_texts_processed)} texts")

In [ ]:
# TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

print("Creating TF-IDF vectorizer...")
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    stop_words='english'
)

X = vectorizer.fit_transform(train_texts_processed)
print(f"\nVectorization results:")
print(f"  Shape: {X.shape}")
print(f"  Number of features: {len(vectorizer.get_feature_names_out())}")
print(f"  Sparsity: {1.0 - (X.nnz / (X.shape[0] * X.shape[1]))} (higher = more sparse)")

In [ ]:
# Most important features (TF-IDF weights) by class
print("Top 10 features by disease class:")
print("="*60)

for label_id in sorted(label_map.keys()):
    mask = train_df['condition_label'] == label_id
    class_mean_tfidf = X[mask].mean(axis=0).A1
    top_indices = class_mean_tfidf.argsort()[-10:][::-1]
    
    feature_names = vectorizer.get_feature_names_out()
    top_features = [feature_names[i] for i in top_indices]
    
    print(f"\n{label_map[label_id]}:")
    print(f"  {', '.join(top_features)}")

## 5. Model Training and Evaluation

In [ ]:
# Import model training module
from src.model.train import MedicalTextClassifier
from sklearn.model_selection import train_test_split

# Prepare data
X = vectorizer.fit_transform(train_texts_processed)
y = train_df['condition_label'].values

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"\nClass distribution in training set:")
for label_id, count in np.bincount(y_train):
    print(f"  {label_map[label_id]}: {count}")

In [ ]:
# Train Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Training Random Forest classifier...")
classifier = RandomForestClassifier(
    n_estimators=150,
    max_depth=25,
    random_state=42,
    n_jobs=-1,
    verbose=0
)

classifier.fit(X_train, y_train)
print("Training complete!")

In [ ]:
# Model evaluation
y_pred = classifier.predict(X_test)
y_proba = classifier.predict_proba(X_test)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")

# Classification report
class_names = [label_map[i] for i in sorted(label_map.keys())]
clsf_report = classification_report(y_test, y_pred, target_names=class_names)
print("\nClassification Report:")
print(clsf_report)

In [ ]:
# Confusion matrix visualization
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            ax=ax, cbar_kws={'label': 'Count'})
ax.set_title('Confusion Matrix - Test Set', fontsize=12, fontweight='bold')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('../data/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Model Serialization and Unit Testing

In [ ]:
# Test inference with sample texts
test_samples = [
    "Patient diagnosed with advanced lung cancer requiring immediate chemotherapy",
    "Severe gastric ulcer causing persistent abdominal pain and bleeding",
    "Neurological disorder causing tremors and loss of motor control",
    "Cardiac arrhythmia detected during routine ECG examination",
    "Patient presents with high fever and signs of systemic infection"
]

print("Testing inference on sample texts:")
print("="*70)

for i, text in enumerate(test_samples, 1):
    # Preprocess
    processed = preprocessor.preprocess(text)
    # Vectorize
    X_sample = vectorizer.transform([processed])
    # Predict
    pred_label = classifier.predict(X_sample)[0]
    pred_proba = classifier.predict_proba(X_sample)[0].max()
    
    print(f"\n{i}. Text: {text[:60]}...")
    print(f"   Predicted: {label_map[pred_label]} (confidence: {pred_proba:.3f})")

## 7. Project Summary and Insights

In [ ]:
print("\n" + "="*70)
print("PROJECT SUMMARY")
print("="*70)

print(f"\n📊 Dataset Statistics:")
print(f"  - Training samples: {len(train_df):,}")
print(f"  - Test samples: {len(test_df):,}")
print(f"  - Total classes: {len(label_map)}")
print(f"  - Average text length: {train_df['word_count'].mean():.0f} words")

print(f"\n🤖 Model Configuration:")
print(f"  - Vectorizer: TF-IDF (max_features=5000, ngrams=(1,2))")
print(f"  - Classifier: Random Forest (n_estimators=150, max_depth=25)")
print(f"  - Total features: {X.shape[1]}")

print(f"\n📈 Model Performance:")
print(f"  - Test Accuracy: {accuracy:.4f}")
print(f"  - Model saved to: data/models/classifier_model.pkl")
print(f"  - Vectorizer saved to: data/models/tfidf_vectorizer.pkl")

print(f"\n🚀 Next Steps:")
print(f"  1. Deploy model via FastAPI")
print(f"  2. Build Docker container")
print(f"  3. Setup monitoring (Prometheus + Grafana)")
print(f"  4. Optimize for latency (ONNX export)")
print(f"  5. Configure CI/CD pipeline")